<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 8: Continuing Expressions

This notebook accompanies Chapter 8 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

In Chapter 7 you learned how to *begin* an expression, for example with `pl.col("weight")`. A bare expression only gets you so far. This chapter is about **continuing** an expression by chaining methods onto it: transforming, filling, smoothing, sorting, summarizing, and more. There are well over a hundred such methods, so this notebook doesn't try to memorize them. Instead it teaches you how they are **organized**, so that you can find the right one in the API reference.

By the end of this notebook you will be able to:

- sort any expression method into one of **five families**, based on what it does to the *length* of the data
- apply element-wise math, rounding, and trigonometry, and handle missing, infinite, and `NaN` values
- use cumulative, fill, shift, duplicate, rolling, and sorting operations
- summarize a column with statistics, counts, and quantifiers
- select, deduplicate, and extend values within a column
- test and verify your expressions rather than trusting them

**How to use this notebook:** run the cells from top to bottom. Random data uses a **fixed seed in every cell** that needs it, so you can re-run any cell and get the same numbers.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior. We also install `plotnine` for one plot, and use `%pip` so packages are installed into the running notebook kernel.

In [ ]:
%pip install -q "polars==2.1.*" plotnine

### Imports and helpers

- `download` fetches a data file from the book's repository **only if it isn't already there**, writes to a temporary name and renames it when complete, and raises an error on HTTP failures.
- `timed` runs a function several times and reports the best time, because one timing is unreliable.

**About random numbers.** Several demos use random data. A random generator is *stateful*: each call consumes numbers, so if cells share one generator, the values in a cell depend on which cells ran before. To keep every cell reproducible and independent, we create a **fresh, seeded generator inside each cell** (`np.random.default_rng(1729)`) rather than sharing one.

In [ ]:
import math
from pathlib import Path
from time import perf_counter
import urllib.request

import numpy as np
import plotnine as p9
import polars as pl
from polars.testing import assert_frame_equal

print(f"Polars {pl.__version__}")

BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data"
DATA_DIR = Path("data")


def download(relative_path: str) -> Path:
    """Download BASE_URL/relative_path into DATA_DIR (unless it exists) and return the local path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        urllib.request.urlretrieve(f"{BASE_URL}/{relative_path}", temporary)
        temporary.rename(path)
    return path


timings: list[dict] = []


def timed(label: str, func, repeat: int = 5):
    """Run func() `repeat` times, record the best time, and return the last result."""
    durations = []
    for _ in range(repeat):
        start = perf_counter()
        result = func()
        durations.append(perf_counter() - start)
    timings.append({"approach": label, "best_seconds": round(min(durations), 4)})
    return result

## Types of Operations

Rather than present a long alphabetical list of methods, the chapter groups them by **what they do to the length of the data**. This is the most useful way to think about them, because the length decides where you can use a method and how it combines with other columns.

| Family | Input → output length | Does the order of values matter? | Examples |
|---|---|---|---|
| **Element-wise** | *n* → *n* | No: each value is handled independently | `sqrt`, `round`, `is_null`, `replace` |
| **Series-wise, nonreducing** | *n* → *n* | **Yes**: values depend on their neighbors | `cum_sum`, `shift`, `rolling_mean`, `sort`, `forward fill` |
| **Series-wise, summarize to one** | *n* → **1** | Not usually | `mean`, `max`, `n_unique`, `all`, `first` |
| **Series-wise, summarize to one or more** | *n* → **0 to *n*** | Depends | `unique`, `top_k`, `drop_nulls`, `mode` |
| **Series-wise, extend** | *n* → **more than *n*** | n/a | `explode`, `extend_constant` |

This isn't just a classification; we can **verify it**. Below is one representative of each family, applied to the same 8-value column. We measure the actual output length:

In [ ]:
values_df = pl.DataFrame({"x": [3, 1, 4, 1, 5, 9, 2, 6]})

representatives = {
    "element-wise: sqrt": pl.col("x").sqrt(),
    "series-wise: cum_sum": pl.col("x").cum_sum(),
    "summarize to one: mean": pl.col("x").mean(),
    "summarize to one or more: unique": pl.col("x").unique(),
    "extend: extend_constant": pl.col("x").extend_constant(0, n=3),
}

pl.DataFrame(
    {
        "operation": list(representatives),
        "input_rows": [values_df.height] * len(representatives),
        "output_rows": [values_df.select(expr).height for expr in representatives.values()],
    }
)

Keep this table in mind. It explains the rules you'll run into:

- **Same-length** results can be added next to the original column with `with_columns`.
- A **one-value** result is *repeated* (broadcast) to fill a column. We'll see this below.
- A result with a **different length** can't sit next to the original columns. Inside `select` it works when *all* the outputs have the same length, and inside `group_by(...).agg(...)` each group gets its own result.

### Four quick examples

We use a dataset of penguins. We read it with `null_values="NA"` (Chapter 6) and express body mass in kilograms.

In [ ]:
PENGUINS_CSV = download("penguins.csv")

penguins = pl.read_csv(PENGUINS_CSV, null_values="NA").select(
    "species",
    "island",
    "sex",
    "year",
    mass=pl.col("body_mass_g") / 1000,
)

penguins.head()

**Example A: element-wise.** `sqrt` and `exp` look at one value at a time. The output has the same length as the input, and a `null` simply stays `null`.

In [ ]:
penguins.with_columns(
    mass_sqrt=pl.col("mass").sqrt(),
    mass_exp=pl.col("mass").exp(),
)

**Example B: summarize to one.** `mean` and `first` each reduce the whole column to a single value, so the result has one row.

In [ ]:
penguins.select(pl.col("mass").mean(), pl.col("island").first())

**Example C: summarize to one or more.** `unique` returns each distinct value once, so the number of rows depends on the data. The order of the values isn't guaranteed (Polars works in parallel), so we sort them to make the output stable.

In [ ]:
penguins.select(pl.col("island").unique().sort())

**Example D: extend.** This chain shows how an expression can *change length several times*:

1. `unique()` finds the 3 species (3 values)
2. `repeat_by(3000)` turns each into a *list* of 3,000 copies (still 3 rows, each a list)
3. `explode()` flattens the lists into rows (9,000 values)
4. `extend_constant("Saiyan", n=1)` appends one more value (9,001 values)

The point isn't the result; it's that expressions are flexible enough to build this one-liner.

In [ ]:
(
    penguins.select(
        pl.col("species")
          .unique()
          .repeat_by(3000)
          .explode()
          .extend_constant("Saiyan", n=1)
    ).height
)

## Element-Wise Operations

Element-wise operations handle each value **independently**. The order doesn't matter, and an invalid result for one value (such as `NaN`) never affects the others.

### Mathematical transformations

`abs`, `exp`, `log`, `log10`, `log1p`, `sign`, `sqrt`, and `cbrt`. Arithmetic *between* two expressions, such as adding two columns, is covered in Chapter 9.

Look at the edges of the table below: the log of a negative number is `NaN`, the log of zero is `-inf`, and a huge input overflows to `inf`. Only the affected cells are affected. Notice that `log(2)` takes the base as an argument, while the default is the natural log.

In [ ]:
(
    pl.DataFrame(
        {
            "x": [-2.0, 0.0, 0.5, 1.0, math.e, 1000.0]
        }
    ).with_columns(
        abs=pl.col("x").abs(),
        exp=pl.col("x").exp(),
        log2=pl.col("x").log(2),
        log10=pl.col("x").log10(),
        log1p=pl.col("x").log1p(),
        sign=pl.col("x").sign(),
        sqrt=pl.col("x").sqrt(),
    )
)

### Trigonometry

Polars has `sin`, `cos`, `tan`, their inverses (`arcsin`, `arccos`, `arctan`), their hyperbolic versions, and two converters: `radians()` and `degrees()`. All trigonometric functions work in **radians**, so angles in degrees must be converted first.

(The book's table applies these functions to a mix of radians and degrees, which makes the output hard to interpret. Here we use angles in degrees, convert them, and check the results against values you know.)

In [ ]:
angles_df = pl.DataFrame(
    {
        "degrees": [0.0, 30.0, 45.0, 60.0, 90.0]
    }
).with_columns(
    radians=pl.col("degrees").radians(),
).with_columns(
    sin=pl.col("radians").sin(),
    cos=pl.col("radians").cos(),
).with_columns(
    back_to_degrees=pl.col("sin").arcsin().degrees(),
)

angles_df

Inverse functions only accept inputs in a limited **domain**. `arccos` needs values between -1 and 1, and anything else becomes `NaN`, without affecting the other values:

In [ ]:
pl.DataFrame(
    {
        "x": [-2.0, -1.0, 0.0, 1.0, 2.0]
    }
).select(
    "x",
    arccos=pl.col("x").arccos()
)

### Rounding and categorizing

Real data often has too much precision, or too many distinct values. `round`, `ceil`, `floor` and `clip` limit precision or range, and `bin_intervals` and `bin_quantiles` turn numbers into **categories**:

- `bin_intervals(breaks, labels=...)` uses breakpoints you choose. You need **one more label than breaks** (two breaks make three bins)
- `bin_quantiles(quantiles, labels=...)` picks the breakpoints from the data's **quantiles** (for example, 0.5 is the median)

Notice that rounding to zero decimals keeps a **float** type (`round0` stays `f64`), and that `bin_intervals` and `bin_quantiles` produce a categorical-like column. Look at the schema below to see exactly which type your version produces.

In [ ]:
rounding_df = pl.DataFrame(
    {
        "x": [-6.0, -0.5, 0.0, 0.5, math.pi, 9.9, 9.99, 9.999]
    }
).with_columns(
    ceil=pl.col("x").ceil(),
    clip=pl.col("x").clip(-1, 1),
    cut=pl.col("x").bin_intervals([-1, 1], labels=["bad", "neutral", "good"]),
    floor=pl.col("x").floor(),
    qcut=pl.col("x").bin_quantiles([0.5], labels=["below median", "above median"]),
    round2=pl.col("x").round(2),
    round0=pl.col("x").round(0),
)


In [ ]:
print(rounding_df.schema)

In [ ]:
rounding_df

### Missing and infinite values

Remember from Chapter 4 that `null` (missing) and `NaN` (not a number) are **different**. These methods detect and replace each:

| Detect | Replace |
|---|---|
| `is_null()`, `is_not_null()` | `fill_null(...)` |
| `is_nan()`, `is_not_nan()` | `fill_nan(...)` |
| `is_finite()`, `is_infinite()` | n/a |

Look at the `null` row: `is_finite`, `is_infinite`, and `is_nan` all return `null` there, because a missing value *has no answer* to those questions. Also see that `fill_nan` leaves the `null` alone, and `fill_null` leaves the `NaN` alone.

In [ ]:
special_values = [42.0, math.nan, None, math.inf, -math.inf]

(
    pl.DataFrame(
        {
            "x": special_values
        }
    ).with_columns(
        fill_nan=pl.col("x").fill_nan(999),
        fill_null=pl.col("x").fill_null(0),
        is_finite=pl.col("x").is_finite(),
        is_infinite=pl.col("x").is_infinite(),
        is_nan=pl.col("x").is_nan(),
        is_null=pl.col("x").is_null(),
    )
)

To treat both the same, combine the methods. Chain `fill_nan` and `fill_null`, or detect either one with the OR operator (`|`, Chapter 9). Whether `NaN` and `null` *should* be treated alike depends on your analysis, because they usually mean different things (an invalid computation vs. data that was never collected).

In [ ]:
pl.DataFrame(
    {
        "x": special_values
    }
).with_columns(
    fill_both=pl.col("x").fill_nan(0).fill_null(0),
    is_either=pl.col("x").is_nan() | pl.col("x").is_null(),
)

### Other element-wise operations

- `hash` turns each value into a number. **Do not rely on the specific numbers.** They differ between machines and Polars versions, so use them for grouping or bucketing, not for storing.
- `repeat_by` turns each value into a *list* of repeats.
- `replace` swaps values according to a dictionary.

On `replace`: **values not in the dictionary are left unchanged**, which can hide typos in your mapping. If you want every value mapped, `replace_strict` raises an error for values it doesn't know (or you can give it a `default`). Being strict is usually the better habit.

In [ ]:
words_df = pl.DataFrame({"x": ["here", "there", "their", "they're"]})
mapping = {"here": "there", "they're": "they are"}

words_df.with_columns(
    hash=pl.col("x").hash(seed=1337),
    repeat_by=pl.col("x").repeat_by(3),
    replace=pl.col("x").replace(mapping),
)

In [ ]:
# Strict: "there" and "their" are not in the mapping, so this should raise an error.
words_df.select(pl.col("x").replace_strict(mapping))

In [ ]:
# Strict with an explicit default for everything else.
words_df.with_columns(replaced=pl.col("x").replace_strict(mapping, default="(unmapped)"))

## Nonreducing Series-Wise Operations

From here on, operations look at the **whole Series**, not at one value at a time. The values (and often their *order*) depend on each other. This section covers operations that keep the length the same: *n* values in, *n* values out.

### Operations that accumulate

Cumulative operations walk down the column and keep a running result: `cum_sum`, `cum_prod`, `cum_min`, `cum_max`, and `cum_count`. `diff` is the difference to the previous value, and `pct_change` the relative change. All of them accept `reverse=True` to walk from the bottom.

Pay attention to how `null` and `NaN` behave in the table:

- `cum_count` counts the **non-null** values seen so far
- `cum_sum` and `cum_prod` skip `null` but become `NaN` for the rest of the column once they meet a `NaN`. `NaN` is **contagious**
- `cum_max` and `cum_min` ignore `NaN`
- `diff` and `pct_change` are affected only at the two positions around each `NaN`

In [ ]:
(
    pl.DataFrame(
        {
            "x": [0.0, 1.0, 2.0, None, 2.0, math.nan, -1.0, 2.0]
        }
    ).with_columns(
        cum_count=pl.col("x").cum_count(),
        cum_max=pl.col("x").cum_max(),
        cum_min=pl.col("x").cum_min(),
        cum_prod=pl.col("x").cum_prod(reverse=True),
        cum_sum=pl.col("x").cum_sum(),
        diff=pl.col("x").diff(),
        pct_change=pl.col("x").pct_change(),
    )
)

### Operations that fill and shift

Sometimes a value only makes sense *in the context of its neighbors*:

- **Fill** missing values from the rows around them. We use `fill_null(strategy=...)`, which is the form from Chapter 4: `"forward"` copies the last seen value, `"backward"` the next one, and `limit` caps how many consecutive nulls get filled.
- `interpolate` estimates missing values *between* known ones. `"linear"` draws a straight line between neighbors; `"nearest"` copies the closest known value.
- `shift(n)` moves values down by *n* rows (up if negative), leaving `null` at the end it vacates. This is how you compare each row with the previous one.

Notice that `NaN` is *not* filled or interpolated (it isn't missing).

In [ ]:
(
    pl.DataFrame(
        {
            "x": [-1.0, 0.0, 1.0, None, None, 3.0, 4.0, math.nan, 6.0]
        }
    ).with_columns(
        backward_fill=pl.col("x").fill_null(strategy="backward"),
        forward_fill=pl.col("x").fill_null(strategy="forward", limit=1),
        interp_linear=pl.col("x").interpolate(method="linear"),
        interp_nearest=pl.col("x").interpolate(method="nearest"),
        shift_down=pl.col("x").shift(1),
        shift_up=pl.col("x").shift(-2),
    )
)

The book also uses dedicated `forward_fill()` and `backward_fill()` methods. They should be equivalent to the strategies above.

### Operations related to duplicates

Four methods flag duplicates *without changing the length*, so you can use them as filters:

- `is_duplicated()`: is this value repeated *anywhere* in the column?
- `is_unique()`: does this value appear exactly once?
- `is_first_distinct()` and `is_last_distinct()`: is this the first or last time this value appears?

Note the difference between `is_duplicated` and `is_first_distinct`: for the repeated `"C"`, *both* rows are "duplicated" but only the first (or last) is the first (or last) *distinct* one. Filtering on `is_first_distinct` is a way to keep one row per value.

In [ ]:
(
    pl.DataFrame({"x": ["A", "C", "D", "C"]}).with_columns(
        is_duplicated=pl.col("x").is_duplicated(),
        is_first_distinct=pl.col("x").is_first_distinct(),
        is_last_distinct=pl.col("x").is_last_distinct(),
        is_unique=pl.col("x").is_unique(),
    )
)

### Operations that compute rolling statistics

Rolling statistics slide a **window** along the data and summarize each window. They **smooth** noisy values, such as stock prices. Our example is NVIDIA's closing price for the first half of 2023. Three methods:

- `rolling_mean(window_size=7)`: the average of the current and previous 6 values
- `rolling_min(window_size=7)`: the minimum over the same window
- `ewm_mean(com=7)`: an **exponentially weighted** moving average, which weights recent values more heavily (`com` is the "center of mass" and controls how quickly weights decay)

The first six rolling values are `null`, since the window isn't full yet. (Check the documentation for the argument that allows partial windows.)

First we download just the one file we need:

In [ ]:
NVDA_2023_CSV = download("stock/nvda/2023.csv")

stock = (
    pl.read_csv(NVDA_2023_CSV, try_parse_dates=True)
    .select("date", "close")
    .with_columns(
        ewm_mean=pl.col("close").ewm_mean(com=7, ignore_nulls=True),
        rolling_mean=pl.col("close").rolling_mean(window_size=7),
        rolling_min=pl.col("close").rolling_min(window_size=7),
    )
)

stock

A table makes it hard to see the differences, so let's plot them. We reshape the table with `unpivot` into long format (one row per date and method), which is what plotnine expects.

In [ ]:
(
    p9.ggplot(stock.unpivot(index="date"), p9.aes("date", "value", color="variable"))
    + p9.geom_line(size=1)
    + p9.labs(x="Date", y="Value", color="Method")
    + p9.theme_tufte(base_size=14)
    + p9.theme(figure_size=(8, 5))
)

The rolling mean and the exponential mean smooth the price, but the rolling minimum follows the *lower envelope*. A row-based window of 7 rows isn't "one week" though, since weekends and holidays have no trading rows. For windows defined in **time** (for example the last 7 *days*), use the `*_by` variants, which take a date column. Chapter 13 covers time-based windows in depth.

In [ ]:
stock.select(
    "date",
    "close",
    rolling_7_rows=pl.col("close").rolling_mean(window_size=7),
    rolling_7_days=pl.col("close").rolling_mean_by("date", window_size="7d"),
).head(8)

### Operations that sort

Sorting a *single column* on its own is less common than sorting a whole DataFrame (Chapter 11), because it breaks the link between a value and the rest of its row. That's why these demos show the effect side by side:

- `sort` and `reverse` reorder the values
- `arg_sort` returns the **positions** that would sort the data
- `sort_by("y")` reorders `x` according to the order of **another column**, which keeps the rows' meaning intact
- `rank` assigns ranks, and averages them for ties (the two `3`s share rank 2.5)
- `shuffle(seed=...)` randomly reorders (set the seed so the result is reproducible)

In [ ]:
(
    pl.DataFrame(
        {
            "x": [1, 3, None, 3, 7],
            "y": ["D", "I", "S", "C", "O"],
        }
    ).with_columns(
        arg_sort=pl.col("x").arg_sort(),
        shuffle=pl.col("x").shuffle(seed=7),
        sort=pl.col("x").sort(nulls_last=True),
        sort_by=pl.col("x").sort_by("y"),
        reverse=pl.col("x").reverse(),
        rank=pl.col("x").rank(),
    )
)

### Other: run-length identifiers

`rle_id` gives each *run* of identical consecutive values an ID, starting at 0 and increasing whenever the value changes. This is handy for grouping streaks, for example "consecutive days above a threshold".

In [ ]:
pl.DataFrame({"x": [33, 33, 27, 33, 60, 60, 60, 33, 60]}).with_columns(
    rle_id=pl.col("x").rle_id(),
)

## Series-Wise Operations That Summarize to One

These operations reduce an entire column to **a single value**. What you do next depends on where you use them.

**In `with_columns`**, a one-value result is *repeated* (broadcast) to every row, because all columns must have the same length. That's useful for comparing each value to the average:

In [ ]:
pl.DataFrame({"x": [1, 3, 3, 7]}).with_columns(
    mean=pl.col("x").mean(),
)

**In `group_by(...).agg(...)`**, the summary is computed **once per group**, which is the typical use (Chapter 13):

In [ ]:
(
    pl.DataFrame({"cluster": ["a", "a", "b", "b"], "x": [1, 3, 3, 7]})
    .group_by("cluster")
    .agg(
        mean=pl.col("x").mean(),
    )
    .sort("cluster")
)

In the rest of this section we use `select`, which keeps only the computed columns, so the single values don't get repeated.

### Quantifiers: `all` and `any`

Quantifiers reduce *Booleans* to one Boolean: `all()` is True only if **every** value is True, and `any()` is True if **at least one** is. We apply them to each of three columns (the `pl.all()` that selects all columns is a different thing from the `.all()` method that reduces):

In [ ]:
quantifier_df = pl.DataFrame(
    {
        "x": [True, False, False],
        "y": [True, True, True],
        "z": [False, False, False],
    }
)

quantifier_df.select(
    pl.all().all().name.suffix("_all"),
    pl.all().any().name.suffix("_any"),
)

### Statistics

`min`, `max`, `mean`, `median`, `sum`, `product`, `std`, `var`, `quantile`, `skew`, `kurtosis`, and `entropy` summarize a numeric column. We draw a million samples from a normal distribution with mean 5 and standard deviation 3, using a **seeded** generator, and the estimates should come out near those values. We *check* this rather than just looking at it:

In [ ]:
rng = np.random.default_rng(1729)
samples = rng.normal(loc=5, scale=3, size=1_000_000)

statistics = pl.DataFrame({"x": samples}).select(
    max=pl.col("x").max(),
    mean=pl.col("x").mean(),
    quantile=pl.col("x").quantile(quantile=0.95),
    skew=pl.col("x").skew(),
    std=pl.col("x").std(),
    sum=pl.col("x").sum(),
    var=pl.col("x").var(),
)


In [ ]:
assert math.isclose(statistics["mean"][0], 5, abs_tol=0.05)
assert math.isclose(statistics["std"][0], 3, abs_tol=0.05)
assert abs(statistics["skew"][0]) < 0.05  # a normal distribution is symmetric


In [ ]:
statistics

### Counting

Five methods count things, and they differ in how they treat missing values:

| Method | Counts |
|---|---|
| `len()` | all values, including `null` |
| `count()` | non-null values |
| `null_count()` | `null` values |
| `n_unique()` | distinct values (a `null` counts as a value) |
| `approx_n_unique()` | *approximately* the number of distinct values, but faster |

We create 1,729 random integers and make exactly one of them missing. We use `scatter` to set the value at index 403 to `null`, rather than assigning with brackets (`series[403] = None`), which changes the Series in place and goes against the "don't use brackets" rule.

In [ ]:
rng = np.random.default_rng(1729)
random_ints = pl.Series("x", rng.integers(low=0, high=10_000, size=1_729)).scatter(403, None)

count_df = pl.DataFrame({"x": random_ints}).with_row_index()
count_df.slice(400, 6)

In [ ]:
count_df.select(
    approx_n_unique=pl.col("x").approx_n_unique(),
    count=pl.col("x").count(),
    len=pl.col("x").len(),
    n_unique=pl.col("x").n_unique(),
    null_count=pl.col("x").null_count(),
)

**Exact vs. approximate.** Why would you want an *approximate* count? Because it can be much faster on big data when an estimate is good enough. Let's measure on ten million values: the best of five runs for each method, and the size of the error.

In [ ]:
LARGE_N = 10_000_000  # lower this if your runtime is short on memory

large_df = pl.DataFrame({"x": np.random.default_rng(1729).integers(low=0, high=10_000, size=LARGE_N)})

timings.clear()
exact = timed("n_unique", lambda: large_df.select(pl.col("x").n_unique()).item())
approx = timed("approx_n_unique", lambda: large_df.select(pl.col("x").approx_n_unique()).item())

print(f"exact = {exact:,}, approximate = {approx:,} (error {abs(approx - exact) / exact:.2%})")
pl.DataFrame(timings)

### Other summaries

`first`, `last`, `get(i)` (the value at position *i*), `arg_min`, `arg_max`, `implode` (collect everything into a single list), and `upper_bound` / `lower_bound` (the largest or smallest value the *data type* can hold, not the data).

Note that `get(403)` returns `null` here, because we made that value missing.

In [ ]:
count_df.select(
    arg_min=pl.col("x").arg_min(),
    first=pl.col("x").first(),
    get=pl.col("x").get(403),
    implode=pl.col("x").implode(),
    last=pl.col("x").last(),
    upper_bound=pl.col("x").upper_bound(),
)

## Series-Wise Operations That Summarize to One or More

These return a number of values that **depends on the data**: at least zero, at most *n*.

### Unique values

- `unique()`: the distinct values (order not guaranteed unless `maintain_order=True`, which costs more)
- `unique_counts()`: how often each appears, in order of first appearance
- `arg_unique()`: the *position* of each value's first appearance
- `value_counts()`: both at once, as a **struct** column. Unnest it to get ordinary columns

In [ ]:
unique_df = pl.DataFrame({"x": ["A", "C", "D", "C"]})

unique_df.select(
    arg_unique=pl.col("x").arg_unique(),
    unique=pl.col("x").unique(maintain_order=True),
    unique_counts=pl.col("x").unique_counts(),
)

In [ ]:
unique_df.select(pl.col("x").value_counts(sort=True)).unnest("x")

### Selecting values

Pick values by position or rank: `head`, `tail`, `slice`, `gather` (specific positions), `gather_every` (every *n*th), `sample` (random), and `top_k` / `bottom_k` (the largest or smallest). All the outputs below must have the **same length** (7) to sit in one `select`. For that reason `gather_every` needs a step that yields exactly 7 values (1,729 values ÷ 247 = 7). We compute it instead of hardcoding it, and set the `sample` seed so the output is reproducible.

Two things to notice: `top_k` and `bottom_k` return their values in **no guaranteed order** (sort afterwards if order matters), and `bottom_k` includes the `null`, which counts as the smallest.

In [ ]:
step = count_df.height // 7

count_df.select(
    bottom_k=pl.col("x").bottom_k(7),
    head=pl.col("x").head(7),
    sample=pl.col("x").sample(7, seed=1729),
    slice=pl.col("x").slice(400, 7),
    gather=pl.col("x").gather([1, 1, 2, 3, 5, 8, 13]),
    gather_every=pl.col("x").gather_every(step),
    top_k=pl.col("x").top_k(7),
)

### Dropping missing values

`drop_nulls()` and `drop_nans()` remove the values entirely, so the column gets *shorter*, and (unlike filling) the rows no longer line up with the other columns. Use them in `select` or `agg`, not next to the original columns.

In [ ]:
(
    pl.DataFrame({"x": [None, 1.0, 2.0, 3.0, math.nan]}).select(
        drop_nans=pl.col("x").drop_nans(),
        drop_nulls=pl.col("x").drop_nulls(),
    )
)

### Other operations

A set of one-or-more operations, run on the same list of numbers. We use a separate `select` for each, because they produce different lengths:

- `arg_true()`: the positions where a Boolean expression is True (here, where `x >= 60`)
- `mode()`: the most frequent value(s); there can be a tie
- `rle()`: run-length encoding: the length and value of each run (a **struct**)
- `search_sorted(value)`: where *value* would be inserted to keep a **sorted** column sorted
- `reshape(...)`: rearrange the values into nested rows

In [ ]:
numbers_df = pl.DataFrame({"x": [33, 33, 27, 33, 60, 60, 60, 33, 60]})

numbers_df.select(arg_true=(pl.col("x") >= 60).arg_true())

In [ ]:
numbers_df.select(mode=pl.col("x").mode().sort())

In [ ]:
numbers_df.select(rle=pl.col("x").rle()).unnest("rle")

In [ ]:
numbers_df.select(search_sorted=pl.col("x").sort().search_sorted(42))

In [ ]:
numbers_df.select(reshape=pl.col("x").reshape((3, 3)))

## Series-Wise Operations That Extend

Only two operations make a Series **longer**:

- `explode()` turns a column of lists into one long column (alias: `flatten()`)
- `extend_constant(value, n=...)` appends `n` copies of a value

In [ ]:
pl.DataFrame({"x": [["a", "b"], ["c", "d"]]}).select(explode=pl.col("x").explode())

## Takeaways

- Classify an expression method by what it does to the **length**: element-wise (*n* → *n*, independent), nonreducing Series-wise (*n* → *n*, order matters), summarize to one, summarize to one or more, or extend.
- Element-wise operations never affect each other's results: a `NaN` in one row leaves the others alone.
- `null` (missing) and `NaN` (invalid number) are different. Detect and fill them with different methods, and remember that `NaN` is **contagious** in cumulative operations.
- Rolling windows count *rows*. For windows measured in time, use the `*_by` variants.
- `replace` leaves unknown values alone. Prefer `replace_strict` when every value should be mapped.
- A summary used in `with_columns` is **broadcast** to every row; in `group_by().agg()` it is computed per group.
- `top_k`, `unique`, and `value_counts` don't promise an order. Sort or set `maintain_order` when order matters.
- Make demos **reproducible** (seeded random generators created per cell) and **testable** (assert the properties you expect).

### Next up

In Chapter 9 we learn to **combine** expressions: arithmetic, comparisons, Boolean logic, and conditionals with `when`/`then`/`otherwise`.

### Try it yourself


1. Add a `weight_z_score` column to `penguins`: `(mass - mean) / std`, using expressions only. Check that its mean is about 0 and its std about 1.

2. Using `stock`, compute the **daily percentage change** of `close` with `pct_change`. On which dates did the price rise more than 5%?

3. Use `rle_id` on a Boolean column "close is above its 7-day rolling mean" to find the longest streak of days above the average.

4. Build an `Enum`-typed grade column and map numeric scores to grades with `bin_intervals`. How many labels do you need for 4 breakpoints?

5. Run `df.select(pl.col("x").sample(5))` several times without a seed, then with a seed. What changes?